# Sidekick (week 4)

Course pattern from `4_langgraph/app.py` + `sidekick.py`, plus extra tools (`fetch_url_text`, `append_worklog`, `pretty_json`, `list_sandbox_files`).

Use repo `.venv` (`uv sync`), cwd = this folder. Set `OPENAI_API_KEY` in `.env` at repo root (loaded by `load_dotenv`).

Playwright: `playwright install chromium`, or `SIDEKICK_NO_BROWSER=1`, or missing Chromium → browser tools skipped.


In [38]:
import asyncio
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, Any, Dict, List, Optional
from urllib.parse import urlparse


In [39]:
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit, PlayWrightBrowserToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_experimental.tools import PythonREPLTool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


In [40]:
import gradio as gr
import requests
from dotenv import load_dotenv
from playwright.async_api import async_playwright
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv(override=True)

SANDBOX_DIR = (Path.cwd() / "sandbox").resolve()
SANDBOX_DIR.mkdir(parents=True, exist_ok=True)

pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"
_serper_client = None


In [41]:
def _headless() -> bool:
    return os.getenv("SIDEKICK_HEADLESS", "").strip().lower() in ("1", "true", "yes")


async def playwright_tools():
    if os.getenv("SIDEKICK_NO_BROWSER", "").strip().lower() in ("1", "true", "yes"):
        print("SIDEKICK_NO_BROWSER set: skipping Playwright.")
        return [], None, None
    try:
        playwright = await async_playwright().start()
        browser = await playwright.chromium.launch(headless=_headless())
        toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
        return toolkit.get_tools(), browser, playwright
    except Exception as e:
        hint = str(e).split("\n")[0][:220]
        print(f"Playwright skipped ({hint}). Install: playwright install chromium")
        return [], None, None


def push(text: str) -> str:
    if not pushover_token or not pushover_user:
        return "Pushover not configured (set PUSHOVER_TOKEN and PUSHOVER_USER)."
    try:
        r = requests.post(
            pushover_url,
            data={"token": pushover_token, "user": pushover_user, "message": text},
            timeout=15,
        )
        r.raise_for_status()
    except requests.RequestException as e:
        return f"Push failed: {e}"
    return "ok"


def get_file_tools():
    toolkit = FileManagementToolkit(root_dir=str(SANDBOX_DIR))
    return toolkit.get_tools()


def http_get_text(url: str) -> str:
    limit = 12000
    parsed = urlparse(url.strip())
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        return "Only http/https URLs with a host are allowed."
    try:
        r = requests.get(
            url.strip(),
            timeout=20,
            headers={"User-Agent": "SidekickFetch/1.0"},
        )
    except requests.RequestException as e:
        return f"Request error: {e}"
    text = r.text or ""
    if len(text) > limit:
        text = text[:limit] + f"\n… truncated ({len(r.text)} chars total)"
    return f"status={r.status_code}\n\n{text}"


def worklog_append(line: str) -> str:
    entry = line.replace("\n", " ").replace("\r", "").strip()
    if not entry:
        return "Nothing to append."
    path = SANDBOX_DIR / "worklog.md"
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    block = f"- {ts} {entry}\n"
    with path.open("a", encoding="utf-8") as f:
        f.write(block)
    return f"Appended to {path.name}"


def format_json_string(payload: str) -> str:
    try:
        data = json.loads(payload)
    except json.JSONDecodeError as e:
        return f"Invalid JSON: {e}"
    return json.dumps(data, indent=2, ensure_ascii=False)


def list_sandbox_files(_unused: str = "") -> str:
    cap = 120
    paths: list[str] = []
    for root, dirs, files in os.walk(SANDBOX_DIR):
        dirs.sort()
        for name in sorted(files):
            full = Path(root) / name
            try:
                rel = full.relative_to(SANDBOX_DIR)
            except ValueError:
                continue
            paths.append(str(rel))
            if len(paths) >= cap:
                return "\n".join(paths) + f"\n… stopped at {cap} entries"
    return "\n".join(paths) if paths else "(empty sandbox)"


def _search_or_stub(query: str) -> str:
    global _serper_client
    if not os.getenv("SERPER_API_KEY"):
        return "Web search needs SERPER_API_KEY."
    if _serper_client is None:
        _serper_client = GoogleSerperAPIWrapper()
    return _serper_client.run(query)


async def other_tools():
    push_tool = Tool(
        name="send_push_notification",
        func=push,
        description="Send a short push notification via Pushover when the user explicitly wants an alert on their device.",
    )
    file_tools = get_file_tools()
    search_tool = Tool(
        name="search",
        func=_search_or_stub,
        description="Run a web search and return summarized results.",
    )
    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    python_repl = PythonREPLTool()
    http_tool = Tool(
        name="fetch_url_text",
        func=http_get_text,
        description="GET a public http(s) URL and return status plus body text (truncated).",
    )
    log_tool = Tool(
        name="append_worklog",
        func=worklog_append,
        description="Append one timestamped bullet line to sandbox/worklog.md for milestones or decisions.",
    )
    json_tool = Tool(
        name="pretty_json",
        func=format_json_string,
        description="Parse a JSON string and return a formatted version; errors if invalid.",
    )
    ls_tool = Tool(
        name="list_sandbox_files",
        func=list_sandbox_files,
        description="List up to 120 files under the sandbox directory.",
    )
    return file_tools + [
        push_tool,
        search_tool,
        wiki,
        python_repl,
        http_tool,
        log_tool,
        json_tool,
        ls_tool,
    ]


In [42]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )


class Sidekick:
    def __init__(self):
        self.worker_llm_with_tools = None
        self.evaluator_llm_with_output = None
        self.tools = None
        self.llm_with_tools = None
        self.graph = None
        self.sidekick_id = str(uuid.uuid4())
        self.memory = MemorySaver()
        self.browser = None
        self.playwright = None

    async def setup(self):
        self.tools, self.browser, self.playwright = await playwright_tools()
        self.tools += await other_tools()
        worker_llm = ChatOpenAI(model="gpt-4o-mini")
        self.worker_llm_with_tools = worker_llm.bind_tools(self.tools)
        evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
        self.evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)
        await self.build_graph()

    def worker(self, state: State) -> Dict[str, Any]:
        system_message = f"""You are a helpful assistant that can use tools to complete tasks.
    You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
    You have many tools to help you, including tools to browse the internet, navigating and retrieving web pages.
    You have a tool to run python code, but note that you would need to include a print() statement if you wanted to receive output.
    You also have search, sandbox file tools, Pushover when asked, fetch_url_text, append_worklog, pretty_json, and list_sandbox_files.
    The current date and time is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

    This is the success criteria:
    {state["success_criteria"]}
    You should reply either with a question for the user about this assignment, or with your final response.
    If you have a question for the user, you need to reply by clearly stating your question. An example might be:

    Question: please clarify whether you want a summary or a detailed answer

    If you've finished, reply with the final answer, and don't ask a question; simply reply with the answer.
    """

        if state.get("feedback_on_work"):
            system_message += f"""
    Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
    Here is the feedback on why this was rejected:
    {state["feedback_on_work"]}
    With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user."""

        found_system_message = False
        messages = state["messages"]
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True

        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages

        response = self.worker_llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def worker_router(self, state: State) -> str:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        else:
            return "evaluator"

    def format_conversation(self, messages: List[Any]) -> str:
        conversation = "Conversation history:\n\n"
        for message in messages:
            if isinstance(message, HumanMessage):
                conversation += f"User: {message.content}\n"
            elif isinstance(message, AIMessage):
                text = message.content or "[Tools use]"
                conversation += f"Assistant: {text}\n"
        return conversation

    def evaluator(self, state: State) -> State:
        last_response = state["messages"][-1].content

        system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
    Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
    and whether more input is needed from the user."""

        user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

    The entire conversation with the assistant, with the user's original request and all replies, is:
    {self.format_conversation(state["messages"])}

    The success criteria for this assignment is:
    {state["success_criteria"]}

    And the final response from the Assistant that you are evaluating is:
    {last_response}

    Respond with your feedback, and decide if the success criteria is met by this response.
    Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.

    The Assistant has access to a tool to write files. If the Assistant says they have written a file, then you can assume they have done so.
    Overall you should give the Assistant the benefit of the doubt if they say they've done something. But you should reject if you feel that more work should go into this.

    """
        if state["feedback_on_work"]:
            user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
            user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."

        eval_result = self.evaluator_llm_with_output.invoke(
            [
                SystemMessage(content=system_message),
                HumanMessage(content=user_message),
            ]
        )
        return {
            "messages": [
                {
                    "role": "assistant",
                    "content": f"Evaluator Feedback on this answer: {eval_result.feedback}",
                }
            ],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
        }

    def route_based_on_evaluation(self, state: State) -> str:
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        else:
            return "worker"

    async def build_graph(self):
        graph_builder = StateGraph(State)
        graph_builder.add_node("worker", self.worker)
        graph_builder.add_node("tools", ToolNode(tools=self.tools))
        graph_builder.add_node("evaluator", self.evaluator)
        graph_builder.add_conditional_edges(
            "worker", self.worker_router, {"tools": "tools", "evaluator": "evaluator"}
        )
        graph_builder.add_edge("tools", "worker")
        graph_builder.add_conditional_edges(
            "evaluator", self.route_based_on_evaluation, {"worker": "worker", "END": END}
        )
        graph_builder.add_edge(START, "worker")
        self.graph = graph_builder.compile(checkpointer=self.memory)

    async def run_superstep(self, message, success_criteria, history):
        config = {"configurable": {"thread_id": self.sidekick_id}}

        state = {
            "messages": message,
            "success_criteria": success_criteria or "The answer should be clear and accurate",
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
        }
        result = await self.graph.ainvoke(state, config=config)
        user = {"role": "user", "content": message}
        reply = {"role": "assistant", "content": result["messages"][-2].content}
        feedback = {"role": "assistant", "content": result["messages"][-1].content}
        return history + [user, reply, feedback]

    def cleanup(self):
        if self.browser:
            try:
                loop = asyncio.get_running_loop()
                loop.create_task(self.browser.close())
                if self.playwright:
                    loop.create_task(self.playwright.stop())
            except RuntimeError:
                asyncio.run(self.browser.close())
                if self.playwright:
                    asyncio.run(self.playwright.stop())


In [ ]:
async def setup():
    sidekick = Sidekick()
    await sidekick.setup()
    return sidekick


async def process_message(sidekick, message, success_criteria, history):
    history = history or []
    text = (message or "").strip()
    if not text:
        yield history, sidekick, message
        return
    if sidekick is None:
        sidekick = Sidekick()
        await sidekick.setup()
    pending = history + [
        {"role": "user", "content": text},
        {"role": "assistant", "content": "Working…"},
    ]
    yield pending, sidekick, ""
    results = await sidekick.run_superstep(text, success_criteria, history)
    yield results, sidekick, ""


async def reset():
    new_sidekick = Sidekick()
    await new_sidekick.setup()
    return "", "", None, new_sidekick


def free_resources(sidekick):
    print("Cleaning up")
    try:
        if sidekick:
            sidekick.cleanup()
    except Exception as e:
        print(f"Exception during cleanup: {e}")


with gr.Blocks(title="Sidekick", theme=gr.themes.Default(primary_hue="emerald")) as ui:
    gr.Markdown("## Sidekick Personal Co-Worker")
    sidekick = gr.State(delete_callback=free_resources)

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False, placeholder="What are your success critiera?"
            )
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")

    ui.load(setup, [], [sidekick])
    message.submit(
        process_message,
        [sidekick, message, success_criteria, chatbot],
        [chatbot, sidekick, message],
    )
    success_criteria.submit(
        process_message,
        [sidekick, message, success_criteria, chatbot],
        [chatbot, sidekick, message],
    )
    go_button.click(
        process_message,
        [sidekick, message, success_criteria, chatbot],
        [chatbot, sidekick, message],
    )
    reset_button.click(reset, [], [message, success_criteria, chatbot, sidekick])


ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Playwright skipped (BrowserType.launch: Executable doesn't exist at /Users/hakeemerisan/Library/Caches/ms-playwright/chromium-1187/chrome-mac/Chromium.app/Contents/MacOS/Chromium). Install: playwright install chromium
Playwright skipped (BrowserType.launch: Executable doesn't exist at /Users/hakeemerisan/Library/Caches/ms-playwright/chromium-1187/chrome-mac/Chromium.app/Contents/MacOS/Chromium). Install: playwright install chromium
